In [2]:
from two_tower_confounding.models.towers import *
from two_tower_confounding.metrics import NDCG, MRR, NegativeLogLikelihood
from two_tower_confounding.models.two_tower import TwoTowerModel
from two_tower_confounding.simulation.simulator import Simulator
from two_tower_confounding.trainer import Trainer
from two_tower_confounding.utils import np_collate

/opt/homebrew/Caskroom/miniforge/base/envs/two-tower-confounding-2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [21]:
# load: two-tower-confounding/results/test/test_datasets/test_click_dataset_policy_temperature0.0_sdoc0.0_num_queries1_.pkl

import pickle as pkl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import two_tower_confounding as ttc

with open('results/test/test_datasets/test_click_dataset_policy_temperature0.0_sdoc0.0_num_queries1_.pkl', 'rb') as f:
    click_data = pkl.load(f)
with open('results/test/test_datasets/test_click_dataset_policy_temperature1.0_sdoc0.0_num_queries1_.pkl', 'rb') as f:
    click_data_tmp_1 = pkl.load(f)


In [22]:
df = pd.DataFrame(list(click_data))
df_tmp_1 = pd.DataFrame(list(click_data_tmp_1))

In [23]:
df_tmp_1

,query,query_doc_features,query_doc_ids,labels,propensities,clicks,positions,mask,n
0,0,"[[0.0, 0.91880757], [0.0, 1.5281845], [0.0, 1....","[8, 3, 1, 10, 2, 5, 9, 4, 7, 6]","[3.0331512250193753, 0.4196948181788126, 0.0, ...","[0.09972, 0.10084, 0.102, 0.09958, 0.0994, 0.1...","[1, 0, 0, 0, 0, 0, 1, 0, 0, 0]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10
1,0,"[[0.0, 1.2311307], [0.0, 1.741676], [0.0, 1.56...","[6, 1, 2, 4, 9, 3, 5, 8, 10, 7]","[1.543456395557677, 0.0, 0.2965051492107924, 0...","[0.10166, 0.10114, 0.10034, 0.09934, 0.09934, ...","[0, 0, 0, 0, 1, 0, 1, 0, 1, 1]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10
2,0,"[[0.0, 0.8156303], [0.0, 1.5281845], [0.0, 1.4...","[9, 3, 4, 5, 6, 8, 7, 1, 10, 2]","[3.564921508576788, 0.4196948181788126, 0.8042...","[0.10142, 0.10084, 0.09852, 0.10014, 0.10088, ...","[1, 0, 0, 0, 0, 1, 0, 0, 0, 0]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10
3,0,"[[0.0, 0.8156303], [0.0, 0.16347846], [0.0, 0....","[9, 10, 7, 8, 5, 6, 4, 3, 1, 2]","[3.564921508576788, 4.0, 2.8558281255060765, 3...","[0.10142, 0.1022, 0.10012, 0.1016, 0.10014, 0....","[1, 1, 0, 1, 0, 0, 0, 0, 0, 0]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10
4,0,"[[0.0, 1.5281845], [0.0, 0.91880757], [0.0, 1....","[3, 8, 2, 10, 1, 6, 5, 4, 7, 9]","[0.4196948181788126, 3.0331512250193753, 0.296...","[0.10122, 0.09886, 0.10034, 0.09958, 0.10034, ...","[0, 0, 0, 1, 0, 0, 0, 1, 1, 1]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10
...,...,...,...,...,...,...,...,...,...
49995,0,"[[0.0, 1.2311307], [0.0, 0.95327824], [0.0, 1....","[6, 7, 5, 2, 9, 1, 3, 10, 4, 8]","[1.543456395557677, 2.8558281255060765, 0.9587...","[0.10166, 0.09906, 0.10166, 0.09918, 0.09934, ...","[0, 1, 0, 0, 0, 0, 0, 1, 0, 0]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10
49996,0,"[[0.0, 1.5281845], [0.0, 0.91880757], [0.0, 0....","[3, 8, 9, 6, 1, 4, 10, 2, 5, 7]","[0.4196948181788126, 3.0331512250193753, 3.564...","[0.10122, 0.09886, 0.10076, 0.10056, 0.10034, ...","[1, 1, 1, 0, 0, 0, 0, 0, 0, 0]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10
49997,0,"[[0.0, 1.5654233], [0.0, 0.91880757], [0.0, 0....","[2, 8, 7, 5, 9, 1, 6, 10, 3, 4]","[0.2965051492107924, 3.0331512250193753, 2.855...","[0.09916, 0.09886, 0.10012, 0.10014, 0.09934, ...","[0, 1, 1, 0, 1, 0, 0, 1, 0, 0]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10
49998,0,"[[0.0, 1.3768247], [0.0, 1.2311307], [0.0, 0.9...","[5, 6, 7, 8, 2, 10, 1, 3, 9, 4]","[0.958746394850535, 1.543456395557677, 2.85582...","[0.09862, 0.09832, 0.10012, 0.1016, 0.0994, 0....","[0, 0, 0, 1, 0, 1, 0, 0, 0, 0]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10


In [24]:
df.query_doc_ids = df.query_doc_ids.apply(lambda x: tuple(x))
df.query_doc_ids.unique()

array([(np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10))],
      dtype=object)

In [25]:
df

,query,query_doc_features,query_doc_ids,labels,propensities,clicks,positions,mask,n
0,0,"[[0.0, 1.741676], [0.0, 1.5654233], [0.0, 1.52...","(1, 2, 3, 4, 5, 6, 7, 8, 9, 10)","[0.0, 0.2965051492107924, 0.4196948181788126, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[0, 0, 0, 0, 0, 0, 1, 0, 0, 0]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10
1,0,"[[0.0, 1.741676], [0.0, 1.5654233], [0.0, 1.52...","(1, 2, 3, 4, 5, 6, 7, 8, 9, 10)","[0.0, 0.2965051492107924, 0.4196948181788126, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[0, 0, 0, 0, 0, 0, 1, 0, 0, 1]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10
2,0,"[[0.0, 1.741676], [0.0, 1.5654233], [0.0, 1.52...","(1, 2, 3, 4, 5, 6, 7, 8, 9, 10)","[0.0, 0.2965051492107924, 0.4196948181788126, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10
3,0,"[[0.0, 1.741676], [0.0, 1.5654233], [0.0, 1.52...","(1, 2, 3, 4, 5, 6, 7, 8, 9, 10)","[0.0, 0.2965051492107924, 0.4196948181788126, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10
4,0,"[[0.0, 1.741676], [0.0, 1.5654233], [0.0, 1.52...","(1, 2, 3, 4, 5, 6, 7, 8, 9, 10)","[0.0, 0.2965051492107924, 0.4196948181788126, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[0, 0, 0, 0, 0, 0, 1, 1, 1, 1]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10
...,...,...,...,...,...,...,...,...,...
49995,0,"[[0.0, 1.741676], [0.0, 1.5654233], [0.0, 1.52...","(1, 2, 3, 4, 5, 6, 7, 8, 9, 10)","[0.0, 0.2965051492107924, 0.4196948181788126, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[0, 0, 0, 0, 0, 0, 0, 1, 0, 0]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10
49996,0,"[[0.0, 1.741676], [0.0, 1.5654233], [0.0, 1.52...","(1, 2, 3, 4, 5, 6, 7, 8, 9, 10)","[0.0, 0.2965051492107924, 0.4196948181788126, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 1, 0]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10
49997,0,"[[0.0, 1.741676], [0.0, 1.5654233], [0.0, 1.52...","(1, 2, 3, 4, 5, 6, 7, 8, 9, 10)","[0.0, 0.2965051492107924, 0.4196948181788126, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[0, 0, 1, 0, 1, 0, 0, 1, 1, 1]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10
49998,0,"[[0.0, 1.741676], [0.0, 1.5654233], [0.0, 1.52...","(1, 2, 3, 4, 5, 6, 7, 8, 9, 10)","[0.0, 0.2965051492107924, 0.4196948181788126, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[0, 0, 0, 0, 0, 0, 0, 1, 0, 0]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10


In [18]:
df_tmp_1.query_doc_ids = df.query_doc_ids.apply(lambda x: tuple(x))
df_tmp_1.query_doc_ids.unique()

array([(np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10))],
      dtype=object)

In [19]:
df_tmp_1.query_doc_ids

0    (1, 2, 3, 4, 5, 6, 7, 8, 9, 10)
Name: query_doc_ids, dtype: object

In [68]:
df

,query,query_doc_features,query_doc_ids,labels,propensities,clicks,positions,mask,n
0,4,"[[0.7015617, 0.5488148, 0.60201824, 0.24623486...","(41, 42, 43, 44, 45, 46, 47, 48, 49, 50)","[1.3039911655389678, 1.3039911655389678, 1.303...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10
1,8,"[[-0.89641446, 0.5857847, -0.2175278, 0.133393...","(81, 82, 83, 84, 85, 86, 87, 88, 89, 90)","[1.8963089162004436, 1.8963089162004436, 1.896...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[1, 1, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10
2,1,"[[-0.3590721, 0.67289346, -0.15888232, -0.1986...","(11, 12, 13, 14, 15, 16, 17, 18, 19, 20)","[2.585823372687536, 2.585823372687536, 2.58582...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[1, 0, 0, 1, 1, 0, 0, 0, 1, 1]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10
3,1,"[[-0.3590721, 0.67289346, -0.15888232, -0.1986...","(11, 12, 13, 14, 15, 16, 17, 18, 19, 20)","[2.585823372687536, 2.585823372687536, 2.58582...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[0, 0, 1, 0, 1, 0, 0, 0, 1, 0]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10
4,8,"[[-0.89641446, 0.5857847, -0.2175278, 0.133393...","(81, 82, 83, 84, 85, 86, 87, 88, 89, 90)","[1.8963089162004436, 1.8963089162004436, 1.896...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10
...,...,...,...,...,...,...,...,...,...
49995,1,"[[-0.3590721, 0.67289346, -0.15888232, -0.1986...","(11, 12, 13, 14, 15, 16, 17, 18, 19, 20)","[2.585823372687536, 2.585823372687536, 2.58582...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[1, 1, 0, 0, 0, 1, 0, 1, 1, 1]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10
49996,2,"[[0.21244654, -0.43061292, 0.25278002, 0.48217...","(21, 22, 23, 24, 25, 26, 27, 28, 29, 30)","[2.8512289886567426, 2.8512289886567426, 2.851...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[1, 0, 0, 0, 0, 0, 1, 0, 1, 0]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10
49997,7,"[[0.018036364, -0.33590063, -0.24537462, -0.38...","(71, 72, 73, 74, 75, 76, 77, 78, 79, 80)","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10
49998,8,"[[-0.89641446, 0.5857847, -0.2175278, 0.133393...","(81, 82, 83, 84, 85, 86, 87, 88, 89, 90)","[1.8963089162004436, 1.8963089162004436, 1.896...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10
